# Synthetic Medical Image Generation Using GANs for Data Augmentation
**Base Paper:** Myong et al. (2023), *PLOS ONE* (Q1 Journal, SJR ~0.85)  
**Dataset:** Chest X-Ray Images (Pneumonia/Normal) by Kermany et al.

This notebook executes all 3 experiments on Kaggle NVIDIA GPU:
1. **Experiment 1:** Baseline DCGAN (Raw Medical Scans)
2. **Experiment 2:** Preprocessing 1 - Contrast Limited Adaptive Histogram Equalization (CLAHE)
3. **Experiment 3:** Preprocessing 2 - Gaussian Unsharp Masking (High-Frequency Sharpening)

In [ ]:
# 1. Environment & GPU Verification
import os
import glob
import cv2
import json
import numpy as np
import scipy.linalg
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.utils import save_image, make_grid
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(0)/(1024**2):.1f} MB")

In [ ]:
# 2. Locate Chest X-Ray Dataset on Kaggle
possible_paths = [
    '/kaggle/input/chest-xray-pneumonia/chest_xray/train',
    '/kaggle/input/chest-xray-pneumonia/chest_xray/chest_xray/train',
    '/kaggle/input/chest-xray-pneumonia'
]
DATA_DIR = None
for p in possible_paths:
    if os.path.exists(p) and len(glob.glob(os.path.join(p, '**', '*.jpeg'), recursive=True)) > 0:
        DATA_DIR = p
        break

if DATA_DIR:
    images_found = len(glob.glob(os.path.join(DATA_DIR, '**', '*.jpeg'), recursive=True))
    print(f"Dataset found at: {DATA_DIR}")
    print(f"Total real X-ray images available: {images_found}")
else:
    print("Kaggle input dataset not found. Please attach 'chest-xray-pneumonia' via '+ Add Input'. (Simulation mode will be used if absent).")

In [ ]:
# 3. Preprocessing Modules (CLAHE & Unsharp Masking)
class CLAHEPreprocessing:
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    def __call__(self, img_np):
        if len(img_np.shape) == 2:
            return self.clahe.apply(img_np)
        elif len(img_np.shape) == 3:
            lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
            l, a, b = cv2.split(lab)
            l = self.clahe.apply(l)
            return cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB)
        return img_np

class UnsharpMaskPreprocessing:
    def __init__(self, kernel_size=(5, 5), sigma=1.0, amount=1.5):
        self.kernel_size = kernel_size
        self.sigma = sigma
        self.amount = amount
    def __call__(self, img_np):
        blurred = cv2.GaussianBlur(img_np, self.kernel_size, self.sigma)
        mask = img_np.astype(np.float32) - blurred.astype(np.float32)
        enhanced = img_np.astype(np.float32) + self.amount * mask
        return np.clip(enhanced, 0, 255).astype(np.uint8)

In [ ]:
# 4. PyTorch Dataset Loader with Preprocessing Pipeline
class KaggleMedicalDataset(Dataset):
    def __init__(self, root_dir=None, img_size=64, mode='baseline', max_samples=1200):
        self.img_size = img_size
        self.mode = mode.lower()
        self.clahe_filter = CLAHEPreprocessing()
        self.unsharp_filter = UnsharpMaskPreprocessing()
        
        self.image_paths = []
        self.labels = []
        if root_dir and os.path.exists(root_dir):
            for label, cls_name in enumerate(['NORMAL', 'PNEUMONIA']):
                cls_folder = os.path.join(root_dir, cls_name)
                if os.path.exists(cls_folder):
                    files = glob.glob(os.path.join(cls_folder, '*.jpeg')) + glob.glob(os.path.join(cls_folder, '*.png'))
                    self.image_paths.extend(files[:max_samples//2])
                    self.labels.extend([label] * len(files[:max_samples//2]))
                    
        self.use_simulation = (len(self.image_paths) == 0)
        self.count = len(self.image_paths) if not self.use_simulation else max_samples

    def _simulated_radiograph(self, idx):
        np.random.seed(idx)
        img = np.zeros((self.img_size, self.img_size), dtype=np.uint8)
        cv2.ellipse(img, (self.img_size//2, self.img_size//2), (int(self.img_size*0.42), int(self.img_size*0.46)), 0, 0, 360, 110, -1)
        cv2.ellipse(img, (int(self.img_size*0.35), int(self.img_size*0.5)), (int(self.img_size*0.16), int(self.img_size*0.32)), 5, 0, 360, 40, -1)
        cv2.ellipse(img, (int(self.img_size*0.65), int(self.img_size*0.5)), (int(self.img_size*0.16), int(self.img_size*0.32)), -5, 0, 360, 40, -1)
        cv2.ellipse(img, (int(self.img_size*0.52), int(self.img_size*0.58)), (int(self.img_size*0.14), int(self.img_size*0.18)), -15, 0, 360, 130, -1)
        for r in range(int(self.img_size*0.25), int(self.img_size*0.8), int(self.img_size*0.12)):
            cv2.ellipse(img, (self.img_size//2, r), (int(self.img_size*0.36), int(self.img_size*0.08)), 0, 0, 180, 140, 2)
        if idx % 2 == 1:
            cv2.circle(img, (int(self.img_size*0.35), int(self.img_size*0.55)), int(self.img_size*0.12), 95, -1)
        noise = np.random.normal(0, 5, img.shape).astype(np.int16)
        return np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)

    def __len__(self):
        return self.count

    def __getitem__(self, idx):
        if not self.use_simulation:
            img = cv2.imread(self.image_paths[idx], cv2.IMREAD_GRAYSCALE)
            img = cv2.resize(img, (self.img_size, self.img_size))
            label = self.labels[idx]
        else:
            img = self._simulated_radiograph(idx)
            label = 1 if (idx % 2 == 1) else 0

        if self.mode == 'clahe':
            img = self.clahe_filter(img)
        elif self.mode == 'unsharp':
            img = self.unsharp_filter(img)

        img = np.expand_dims(img, axis=0)
        tensor_img = (torch.from_numpy(img).float() / 127.5) - 1.0
        return tensor_img, label

In [ ]:
# 5. DCGAN Architecture (Radford et al. & Myong et al., 2023)
def weights_init_normal(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm2d') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

class Generator(nn.Module):
    def __init__(self, latent_dim=100, channels=1, ngf=64):
        super().__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8), nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4), nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2), nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf), nn.ReLU(True),
            nn.ConvTranspose2d(ngf, channels, 4, 2, 1, bias=False),
            nn.Tanh()
        )
        self.apply(weights_init_normal)
    def forward(self, z):
        if len(z.shape) == 2:
            z = z.view(z.size(0), z.size(1), 1, 1)
        return self.main(z)

class Discriminator(nn.Module):
    def __init__(self, channels=1, ndf=64):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(channels, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False)
        )
        self.apply(weights_init_normal)
    def forward(self, img):
        return self.main(img).view(-1, 1)

In [ ]:
# 6. Generative Evaluation (FID) & Downstream Classifier (Myong et al., 2023)
def calculate_fid(real_feats, gen_feats, eps=1e-5):
    mu1, sigma1 = np.mean(real_feats, axis=0), np.cov(real_feats, rowvar=False) + np.eye(real_feats.shape[1])*eps
    mu2, sigma2 = np.mean(gen_feats, axis=0), np.cov(gen_feats, rowvar=False) + np.eye(gen_feats.shape[1])*eps
    diff = mu1 - mu2
    covmean, _ = scipy.linalg.sqrtm(sigma1.dot(sigma2), disp=False)
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    return float(diff.dot(diff) + np.trace(sigma1) + np.trace(sigma2) - 2 * np.trace(covmean))

class FeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, 2, 1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2, 1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten()
        )
    def forward(self, x): return self.net(x)

class DownstreamClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.AdaptiveAvgPool2d((4, 4)), nn.Flatten(),
            nn.Linear(64*4*4, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 2)
        )
    def forward(self, x): return self.net(x)

In [ ]:
# 7. Master Training Loop across Experiments 1, 2, and 3
OUTPUT_DIR = '/kaggle/working/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

EXPERIMENTS = [
    ('experiment_1_baseline', 'baseline', 'Exp 1: Baseline DCGAN (Raw)'),
    ('experiment_2_clahe', 'clahe', 'Exp 2: CLAHE Preprocessing'),
    ('experiment_3_unsharp', 'unsharp', 'Exp 3: Unsharp Masking')
]

EPOCHS = 15
BATCH_SIZE = 32
LATENT_DIM = 100
results_dict = {}

for exp_key, mode, title in EXPERIMENTS:
    print(f"\n=======================================================")
    print(f" RUNNING {title.upper()} ON {device}")
    print(f"=======================================================")
    
    dataset = KaggleMedicalDataset(root_dir=DATA_DIR, img_size=64, mode=mode, max_samples=1000)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    
    netG = Generator(latent_dim=LATENT_DIM).to(device)
    netD = Discriminator().to(device)
    
    criterion = nn.BCEWithLogitsLoss()
    optD = optim.Adam(netD.parameters(), lr=0.0002, betas=(0.5, 0.999))
    optG = optim.Adam(netG.parameters(), lr=0.0002, betas=(0.5, 0.999))
    
    d_losses, g_losses = [], []
    fixed_noise = torch.randn(25, LATENT_DIM, 1, 1, device=device)
    
    for epoch in range(1, EPOCHS + 1):
        netG.train(); netD.train()
        e_d_loss, e_g_loss = 0.0, 0.0
        for real_imgs, _ in dataloader:
            real_imgs = real_imgs.to(device)
            b_size = real_imgs.size(0)
            
            # Discriminator
            netD.zero_grad()
            out_r = netD(real_imgs)
            loss_r = criterion(out_r, torch.full((b_size, 1), 0.9, device=device))
            
            noise = torch.randn(b_size, LATENT_DIM, 1, 1, device=device)
            fakes = netG(noise)
            out_f = netD(fakes.detach())
            loss_f = criterion(out_f, torch.full((b_size, 1), 0.0, device=device))
            
            loss_d = loss_r + loss_f
            loss_d.backward()
            optD.step()
            
            # Generator
            netG.zero_grad()
            out_g = netD(fakes)
            loss_g = criterion(out_g, torch.full((b_size, 1), 0.9, device=device))
            loss_g.backward()
            optG.step()
            
            e_d_loss += loss_d.item()
            e_g_loss += loss_g.item()
            
        avg_d = e_d_loss / len(dataloader)
        avg_g = e_g_loss / len(dataloader)
        d_losses.append(avg_d); g_losses.append(avg_g)
        if epoch % 5 == 0 or epoch == EPOCHS:
            print(f"[{exp_key}] Epoch {epoch}/{EPOCHS} | Loss D: {avg_d:.4f} | Loss G: {avg_g:.4f}")

    # Save sample grid
    netG.eval()
    with torch.no_grad():
        samples = (netG(fixed_noise).cpu() + 1.0) / 2.0
        save_image(make_grid(samples, nrow=5, padding=2), f"{OUTPUT_DIR}/{exp_key}_generated_grid.png")
        torch.save(netG.state_dict(), f"{OUTPUT_DIR}/{exp_key}_generator.pth")
        torch.save(netD.state_dict(), f"{OUTPUT_DIR}/{exp_key}_discriminator.pth")
        
    # Save Loss Curve
    plt.figure(figsize=(7, 3.5))
    plt.plot(range(1, EPOCHS+1), d_losses, label='Discriminator Loss', color='crimson')
    plt.plot(range(1, EPOCHS+1), g_losses, label='Generator Loss', color='royalblue')
    plt.title(f"{title} - Loss Dynamics")
    plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.grid(True, alpha=0.3)
    plt.savefig(f"{OUTPUT_DIR}/{exp_key}_loss_curve.png", dpi=150)
    plt.close()
    
    # Calculate FID
    extractor = FeatureExtractor().to(device).eval()
    real_feats, gen_feats = [], []
    with torch.no_grad():
        for imgs, _ in dataloader:
            real_feats.append(extractor(imgs.to(device)).cpu().numpy())
            if len(real_feats)*BATCH_SIZE >= 300: break
        real_feats = np.concatenate(real_feats, axis=0)[:300]
        
        for _ in range(10):
            z = torch.randn(30, LATENT_DIM, 1, 1, device=device)
            gen_feats.append(extractor(netG(z)).cpu().numpy())
        gen_feats = np.concatenate(gen_feats, axis=0)[:300]
        
    fid = calculate_fid(real_feats, gen_feats)
    print(f"[{exp_key}] Final FID: {fid:.2f}")
    
    results_dict[exp_key] = {
        'title': title,
        'final_fid': round(fid, 2),
        'd_loss_final': round(d_losses[-1], 4),
        'g_loss_final': round(g_losses[-1], 4)
    }

with open(f"{OUTPUT_DIR}/experiments_summary.json", 'w') as f:
    json.dump(results_dict, f, indent=4)

In [ ]:
# 8. Visual Summary & Final Comparison Display
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (exp_key, _, title) in zip(axes, EXPERIMENTS):
    img_path = f"{OUTPUT_DIR}/{exp_key}_generated_grid.png"
    if os.path.exists(img_path):
        img = Image.open(img_path)
        ax.imshow(img, cmap='gray')
        ax.axis('off')
        fid_val = results_dict[exp_key]['final_fid']
        ax.set_title(f"{title}\nFID: {fid_val}", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n=======================================================")
print(" ALL 3 EXPERIMENTS COMPLETED ON GPU!")
print(f"All output plots saved to: {OUTPUT_DIR}")
print("=======================================================")
for k, v in results_dict.items():
    print(f"{v['title']:<35} | FID: {v['final_fid']:<6} | Loss D: {v['d_loss_final']:<6} | Loss G: {v['g_loss_final']}")